### 데이터 불러오기

In [1]:
import sys
import os, time, torch, tqdm, gc, numpy as np, pandas as pd

import torch.nn as nn
from torch_geometric.data import HeteroData
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.nn import HeteroConv, SAGEConv
from torch.amp import autocast, GradScaler         
from torch.nn.utils import clip_grad_norm_
import torch.nn.functional as F

from torch_geometric.utils import negative_sampling
from torch.optim.lr_scheduler import ReduceLROnPlateau

from neo4j.exceptions import ServiceUnavailable
from neomodel import config as neomodel_config
from neomodel import db

from app.config.local import NEO4J_HOST, NEO4J_PORT, NEO4J_USER, DB_PASSWORD # src/app/config/local.py
NEO4J_USER     = NEO4J_USER
NEO4J_PWD      = DB_PASSWORD
NEO4J_HOST     = NEO4J_HOST
NEO4J_PORT     = NEO4J_PORT

neomodel_config.DATABASE_URL = f'bolt://{NEO4J_USER}:{NEO4J_PWD}@{NEO4J_HOST}:{NEO4J_PORT}'

try:
    result, _ = db.cypher_query("RETURN 1 AS test")
    print("✅ Neo4j 연결 성공! → 응답값:", result[0][0])
except ServiceUnavailable as e:
    print("❌ Neo4j 연결 실패:", e)
from app.config.local import NEO4J_HOST, NEO4J_PORT, NEO4J_USER, DB_PASSWORD

# 2. Cypher → DataFrame 헬퍼
def cypher_df(q: str, params: dict | None = None) -> pd.DataFrame:
    records, cols = db.cypher_query(q, params or {})
    df = pd.DataFrame(records, columns=cols)
    id_cols = [c for c in df.columns if c.endswith('_id') or c in {'mid', 'oid', 'pid'}]
    for col in id_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64").astype("int64")
    return df

def safe_map(series: pd.Series, mapping: dict[int,int], col: str) -> np.ndarray:
    mapped = series.map(mapping)
    miss   = mapped.isna().sum()
    if miss:
        print(f"⚠️  {col}: {miss:,}개 매핑 실패 → 행 제거")
        mapped = mapped.dropna()
    return mapped.astype(np.int64).to_numpy()

# 3. 전체 Member/Order/Product 인덱스 생성
print("📊 전체 멤버/주문/상품 추출...")
df_member  = cypher_df("MATCH (m:Member) RETURN m.member_id AS mid")
df_order   = cypher_df("MATCH (o:Order)  RETURN o.order_id AS oid")
df_product = cypher_df("MATCH (p:Product) RETURN p.product_id AS pid")
m2i = {mid: i for i, mid in enumerate(df_member.mid.unique())}
o2i = {oid: i for i, oid in enumerate(df_order.oid.unique())}
p2i = {pid: i for i, pid in enumerate(df_product.pid.unique())}
m2i_inv = {v: k for k, v in m2i.items()}
p2i_inv = {v: k for k, v in p2i.items()}
print(f"Total Members: {len(m2i):,} | Orders: {len(o2i):,} | Products: {len(p2i):,}")

# 4. 전체 order, product 피처 재생성
print("📊 피처 로딩 중...")
df_order_feat = cypher_df("""
MATCH (o:Order)
RETURN o.order_id AS oid,
       o.days_since_prior_order_norm AS dsp,
       o.order_dow_norm AS dow,
       o.order_hour_of_day_norm AS hr,
       o.order_number_norm AS idx
""")
df_order_feat["oid_idx"] = safe_map(df_order_feat.oid, o2i, "oid")
order_feat = (df_member  = cypher_df("MATCH (m:Member)  RETURN m.member_id  AS mid")
df_order   = cypher_df("MATCH (o:Order)   RETURN o.order_id   AS oid")
df_product = cypher_df("MATCH (p:Product) RETURN p.product_id AS pid")
    df_order_feat.sort_values("oid_idx")[["dsp","dow","hr","idx"]]
    .to_numpy(np.float32)
)

df_prod_vec = cypher_df("""
MATCH (p:Product)
RETURN p.product_id AS pid,
       p.category_vector + p.name_vector AS vec
""")
df_prod_vec["pid_idx"] = safe_map(df_prod_vec.pid, p2i, "pid")
prod_feat = torch.tensor(
    np.stack(df_prod_vec.sort_values("pid_idx").vec.values),
    dtype=torch.float32
)

print(f"Order feature: {order_feat.shape}")
print(f"Product vector: {prod_feat.shape}")

# 5. 전체 prior 간선 (모든 주문 기준!)
print("📊 구조적 간선 생성...")
df_edge = cypher_df("""
MATCH (m:Member)-[:ORDERED]->(o:Order)-[:CONTAINS]->(p:Product)
RETURN m.member_id AS mid, o.order_id AS oid, p.product_id AS pid
""")
uo = torch.stack((
    torch.from_numpy(safe_map(df_edge.mid, m2i, "mid")),
    torch.from_numpy(safe_map(df_edge.oid, o2i, "oid"))
), 0).to(torch.long)
op = torch.stack((
    torch.from_numpy(safe_map(df_edge.oid, o2i, "oid")),
    torch.from_numpy(safe_map(df_edge.pid, p2i, "pid"))
), 0).to(torch.long)

# 6. HeteroData 전체 그래프 생성 
print("📊 HeteroData 전체 그래프 생성...")
test_data = HeteroData()
test_data['user'].num_nodes    = len(m2i)
test_data['order'].num_nodes   = len(o2i)
test_data['product'].num_nodes = len(p2i)
test_data['user'].x    = nn.Embedding(len(m2i), 32).weight
test_data['order'].x   = torch.tensor(order_feat, dtype=torch.float32)
test_data['product'].x = prod_feat
test_data['user','ordered','order'].edge_index         = uo
test_data['order','ordered_by','user'].edge_index      = uo.flip(0)
test_data['order','contains','product'].edge_index     = op
test_data['product','contained_in','order'].edge_index = op.flip(0)

print("전체 HeteroData 생성 완료.")

✅ config_module: ['CARD_SERVER_ADDRESS', 'DB_HOST', 'DB_NAME', 'DB_PASSWORD', 'DB_PORT', 'DB_USER', 'DEBUG', 'GEMINI_API_KEY', 'GOOGLE_APPLICATION_CREDENTIALS', 'GOOGLE_PROJECT', 'NEO4J_HOST', 'NEO4J_PORT', 'NEO4J_USER', 'SWAGGER_SPECS_JSON_ROUTE', 'SWAGGER_SPECS_ROUTE', 'SWAGGER_STATIC_PATH', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'os']
✅ Neo4j 연결 성공! → 응답값: 1
✅ Neo4j 연결 성공! → 응답값: 1
📊 전체 멤버/주문/상품 추출...
Total Members: 206,209 | Orders: 3,346,083 | Products: 49,685
📊 피처 로딩 중...
Order feature: (3346083, 4)
Product vector: torch.Size([49685, 2048])
📊 구조적 간선 생성...
📊 HeteroData 전체 그래프 생성...
전체 HeteroData 생성 완료.


In [2]:
import gc

# === [여기까지 데이터/그래프 메모리에 업로드 완료] ===
print("🧹 예측에 필요 없는 메모리 해제 중...")

# 1. Cypher로 받은 DataFrame, 매핑에만 사용된 임시 변수 삭제
del df_member, df_order, df_product, df_order_feat, df_prod_vec, df_edge

# 4. Python/torch 가비지 컬렉션 호출
gc.collect()
torch.cuda.empty_cache()
print("✅ 예측에 불필요한 데이터/변수 제거 및 캐시 해제 완료!")


🧹 예측에 필요 없는 메모리 해제 중...
✅ 예측에 불필요한 데이터/변수 제거 및 캐시 해제 완료!


### 예측

In [3]:
# 7. HetSAGE 모델 정의 
class HetSAGE(nn.Module):
    def __init__(self, hid=128, out=128, p_drop=0.2, aggr='sum'):
        super().__init__()
        self.conv1 = HeteroConv({
            ('user', 'ordered', 'order'): SAGEConv((-1, -1), hid, aggr=aggr),
            ('order', 'contains', 'product'): SAGEConv((-1, -1), hid, aggr=aggr),
            ('order', 'ordered_by', 'user'): SAGEConv((-1, -1), hid, aggr=aggr),
            ('product', 'contained_in', 'order'): SAGEConv((-1, -1), hid, aggr=aggr),
        }, aggr=aggr)
        self.conv2 = HeteroConv({
            ('user', 'ordered', 'order'): SAGEConv((hid, hid), out, aggr=aggr),
            ('order', 'contains', 'product'): SAGEConv((hid, hid), out, aggr=aggr),
            ('order', 'ordered_by', 'user'): SAGEConv((hid, hid), out, aggr=aggr),
            ('product', 'contained_in', 'order'): SAGEConv((hid, hid), out, aggr=aggr),
        }, aggr=aggr)
        self.norm1 = nn.LayerNorm(hid)
        self.norm2 = nn.LayerNorm(out)
        self.p_drop = p_drop

    def _drop(self, h: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
        return {k: F.dropout(v, p=self.p_drop, training=self.training) for k, v in h.items()}

    def forward(self, x_dict, edge_index_dict):
        h = self.conv1(x_dict, edge_index_dict)
        h = {k: F.relu(self.norm1(v)) for k, v in h.items()}
        h = self._drop(h)
        h = self.conv2(h, edge_index_dict)
        h = {k: F.relu(self.norm2(v)) for k, v in h.items()}
        h = self._drop(h)
        return h

# 8. 모델 로드
ckpt_path = "checkpoints/best_model.pt"   # 학습 완료된 모델
ckpt = torch.load(ckpt_path, map_location='cpu', weights_only = False)
hid_dim = ckpt.get('hid_dim', 128)
out_dim = ckpt.get('out_dim', 128)
model = HetSAGE(hid=hid_dim, out=out_dim)
model.load_state_dict(ckpt['model_state'])
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
model.eval()

m2i = ckpt['m2i']
o2i = ckpt['o2i']
p2i = ckpt['p2i']
m2i_inv = {v: k for k, v in m2i.items()}
o2i_inv = {v: k for k, v in o2i.items()}
p2i_inv = {v: k for k, v in p2i.items()}

print("✔️  모델 로드 및 준비 완료.")

✔️  모델 로드 및 준비 완료.


In [5]:
@torch.no_grad()
def predict_for_all_members(model, test_data, m2i_inv, p2i_inv, top_k=5, batch_size=200):
    print(f"🔄 전체 유저 {len(m2i_inv):,}명, {batch_size}명씩 배치 예측 시작...")
    predictions = {}
    user_indices = np.arange(len(m2i_inv))
    n_batches = int(np.ceil(len(user_indices) / batch_size))
    test_data = test_data.to(device)
    z = model(test_data.x_dict, test_data.edge_index_dict)
    uemb = z['user'].cpu()
    pemb = z['product'].cpu()
    del z; gc.collect(); torch.cuda.empty_cache()

    for batch_idx in tqdm.tqdm(range(n_batches), desc="🔮 [예측 진행]"):
        start = batch_idx * batch_size
        end = min((batch_idx + 1) * batch_size, len(uemb))
        batch_user_idx = user_indices[start:end]
        for user_idx in batch_user_idx:
            user_id = m2i_inv[user_idx]
            u = uemb[user_idx]
            # top-K 상품 추출
            scores_all, idx_all = [], []
            CHUNK = 100_000
            for s in range(0, len(pemb), CHUNK):
                sc, ix = (pemb[s:s+CHUNK] @ u).topk(top_k)
                scores_all.append(sc)
                idx_all.append(ix + s)
            scores_all = torch.cat(scores_all)
            idx_all = torch.cat(idx_all)
            topk_indices = scores_all.topk(top_k).indices
            top_product_ids = [p2i_inv[idx_all[idx].item()] for idx in topk_indices]
            predictions[user_id] = top_product_ids
    print("🎯 전체 예측 완료!")
    return predictions

predictions = predict_for_all_members(model, test_data, m2i_inv, p2i_inv, top_k=5, batch_size=200)

# 6. (Optional) Neo4j에 저장
def save_predictions_to_neo4j(predictions, model_name="HetSAGE_vFinal"):
    print("📥 Neo4j에 예측결과 저장 중...")
    for member_id, product_ids in tqdm.tqdm(predictions.items(), desc="→ DB 저장"):
        q = """
        MATCH (m:Member {member_id: $member_id})-[:ORDERED]->(o:Order)
        WHERE NOT (o)-[:NEXT]->(:Order)
        WITH m, o ORDER BY o.order_number DESC LIMIT 1
        SET m.predict_order_list = $products,
            m.created_by = $model_name,
            o.predict_order_list = $products,
            o.created_by = $model_name
        """
        db.cypher_query(q, {
            "member_id": str(member_id),
            "products": [str(pid) for pid in product_ids],
            "model_name": model_name
        })
    print("✔️  Neo4j 저장 완료")

# 저장 예시 (선택)
# save_predictions_to_neo4j(predictions, model_name="HetSAGE_vFinal")
print("🧊 파이프라인 종료.")


🔄 전체 유저 206,209명, 200명씩 배치 예측 시작...


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


@torch.no_grad()
def predict_for_all_members_min_mem(model, test_data, m2i_inv, p2i_inv, top_k=5, batch_size=200):
    """
    - 전체 유저를 batch_size 단위로 잘라서 부분 그래프 생성 → 부분 모델 추론 → CPU 결과 저장 → GPU 해제 반복
    - test_data는 전체 그래프(구조적 간선 등 포함)
    - m2i_inv, p2i_inv는 {index: 실제 id}
    """
    device = next(model.parameters()).device
    total_users = test_data['user'].num_nodes
    predictions = {}
    user_indices = np.arange(total_users)
    
    print(f"🔄 전체 유저 {total_users:,}명, {batch_size}명씩 배치 예측 시작...")
    
    for start in tqdm.tqdm(range(0, total_users, batch_size)):
        end = min(start + batch_size, total_users)
        user_batch = user_indices[start:end]
        
        # (1) 부분 그래프 추출 (유저별로 구조적 간선만 잘라내는 함수 필요)
        # 여기선 전체 test_data에서 user batch만 x, 나머지는 전체 (O(N)이라 부담 없음)
        batch_data = HeteroData()
        batch_data['user'].num_nodes = len(user_batch)
        batch_data['order'].num_nodes = test_data['order'].num_nodes
        batch_data['product'].num_nodes = test_data['product'].num_nodes
        batch_data['user'].x = test_data['user'].x[user_batch]
        batch_data['order'].x = test_data['order'].x
        batch_data['product'].x = test_data['product'].x
        # 간선은 전체를 그대로 씀 (속도/메모리상 batch user만 edge filter하는 것도 가능)
        batch_data['user','ordered','order'].edge_index = test_data['user','ordered','order'].clone()
        batch_data['order','ordered_by','user'].edge_index = test_data['order','ordered_by','user'].clone()
        batch_data['order','contains','product'].edge_index = test_data['order','contains','product'].clone()
        batch_data['product','contained_in','order'].edge_index = test_data['product','contained_in','order'].clone()

        batch_data = batch_data.to(device)
        z = model(batch_data.x_dict, batch_data.edge_index_dict)
        uemb, pemb = z['user'].cpu(), z['product'].cpu()
        
        for i, uid in enumerate(user_batch):
            real_uid = m2i_inv[uid]
            u = uemb[i]
            scores_all, idx_all = [], []
            CHUNK = 200_000
            for s in range(0, len(pemb), CHUNK):
                sc, ix = (pemb[s:s+CHUNK] @ u).topk(top_k)
                scores_all.append(sc)
                idx_all.append(ix + s)
            scores_all = torch.cat(scores_all)
            idx_all = torch.cat(idx_all)
            topk_indices = scores_all.topk(top_k).indices
            top_product_ids = [p2i_inv[idx_all[idx].item()] for idx in topk_indices]
            predictions[real_uid] = top_product_ids
        
        # 배치 끝날 때마다 GPU 메모리 정리
        del batch_data, z, uemb, pemb
        torch.cuda.empty_cache()
        gc.collect()

    print("🎯 전체 예측 완료!")
    return predictions

# 모델, test_data, m2i_inv, p2i_inv는 기존과 동일하게 준비해야 함
predictions = predict_for_all_members_min_mem(
    model, test_data, m2i_inv, p2i_inv, top_k=5, batch_size=200
)

# Neo4j 저장 함수 동일하게 사용
def save_predictions_to_neo4j(predictions, model_name="best_model.pt"):
    print("📥 Neo4j에 예측결과 저장 중...")
    for member_id, product_ids in tqdm.tqdm(predictions.items(), desc="→ DB 저장"):
        q = """
        MATCH (m:Member {member_id: $member_id})-[:ORDERED]->(o:Order)
        WHERE NOT (o)-[:NEXT]->(:Order)
        WITH m, o ORDER BY o.order_number DESC LIMIT 1
        SET m.predict_order_list = $products,
            m.created_by = $model_name,
            o.predict_order_list = $products,
            o.created_by = $model_name
        """
        db.cypher_query(q, {
            "member_id": str(member_id),
            "products": [str(pid) for pid in product_ids],
            "model_name": model_name
        })
    print("✔️  Neo4j 저장 완료")

save_predictions_to_neo4j(predictions, model_name="HetSAGE_vFinal")
print("🧊 모든 파이프라인 종료.")